# 📊 Exploratory Data Analysis — AI Smart Pricing Engine
This notebook explores the raw sales data to understand price elasticity, seasonality, and feature relationships before modeling.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path
sys.path.insert(0, str(Path('..').resolve()))

from src.utils.config import RAW_DATA_DIR
from src.visualization.plots import plot_price_vs_demand_scatter, plot_seasonal_patterns, plot_festival_impact

## 1. Data Overview

In [ ]:
df = pd.read_csv(RAW_DATA_DIR / 'sales_data.csv', parse_dates=['date'])
print(f"Dataset Shape: {df.shape}")
display(df.head())
df.info()
display(df.describe())

In [ ]:
print("Missing Values:")
print(df.isnull().sum())

## 2. Univariate Analysis

In [ ]:
fig = px.histogram(df, x="selling_price", nbins=50, marginal="box", 
                   title="Distribution of Selling Price")
fig.show()

In [ ]:
fig = px.histogram(df, x="units_sold", nbins=50, marginal="box", 
                   title="Distribution of Demand (Units Sold)")
fig.show()

In [ ]:
category_sales = df.groupby('product_category')['units_sold'].sum().reset_index()
fig = px.bar(category_sales, x="product_category", y="units_sold", 
             title="Total Sales by Product Category")
fig.show()

In [ ]:
daily_sales = df.groupby('date')['units_sold'].sum().reset_index()
fig = px.line(daily_sales, x="date", y="units_sold", 
              title="Daily Total Units Sold Over Time")
fig.show()

## 3. Bivariate Analysis

In [ ]:
# Scatter plot for a sample product to visualize Price vs Demand
sample_product_id = df['product_id'].iloc[0]
fig = plot_price_vs_demand_scatter(df, product_id=sample_product_id, title=f"Price vs Demand for {sample_product_id}")
fig.show()

In [ ]:
fig = px.scatter(df, x="competitor_price", y="units_sold", color="product_category", 
                 opacity=0.5, trendline="ols", 
                 title="Competitor Price vs Units Sold")
fig.show()

In [ ]:
fig = px.scatter(df, x="rating", y="units_sold", color="product_category", 
                 opacity=0.5, trendline="ols", 
                 title="Product Rating vs Units Sold")
fig.show()

In [ ]:
fig = px.scatter(df, x="ad_spend", y="units_sold", color="product_category", 
                 opacity=0.5, trendline="ols", 
                 title="Ad Spend vs Units Sold")
fig.show()

## 4. Temporal Patterns

In [ ]:
fig = px.box(df, x="day_of_week", y="units_sold", color="product_category", 
             title="Units Sold by Day of Week")
fig.show()

In [ ]:
fig = plot_seasonal_patterns(df, title="Monthly Seasonality Analysis")
fig.show()

In [ ]:
fig = plot_festival_impact(df, title="Festival Impact on Sales")
fig.show()

In [ ]:
weekend_sales = df.groupby('is_weekend')['units_sold'].mean().reset_index()
weekend_sales['is_weekend'] = weekend_sales['is_weekend'].map({0: 'Weekday', 1: 'Weekend'})
fig = px.bar(weekend_sales, x="is_weekend", y="units_sold", 
             title="Average Sales: Weekend vs Weekday")
fig.show()

## 5. Price Elasticity Exploration

In [ ]:
df['log_price'] = np.log(df['selling_price'])
df['log_demand'] = np.log(df['units_sold'] + 1)
fig = px.scatter(df, x="log_price", y="log_demand", opacity=0.3, trendline="ols", 
                 title="Log-Log Plot: Price vs Demand (Overall Elasticity)")
fig.show()

In [ ]:
fig = px.scatter(df, x="log_price", y="log_demand", color="product_category", 
                 opacity=0.3, trendline="ols", 
                 title="Log-Log Plot: Price vs Demand (By Category)")
fig.show()

## 6. Correlation Analysis

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Correlation Heatmap of Numeric Features")
plt.show()

## 7. Key Findings & Insights
- **Price Elasticity**: The log-log plots reveal an inverse relationship between price and demand, confirming price sensitivity.
- **Seasonality**: Demand spikes during specific months and on weekends.
- **Festivals**: Significant uplift in sales is observed during festival periods.
- **Competitor Influence**: Competitor pricing and ad spend positively correlate with demand, showing external market effects.